Some Versions of libraries to download, so errors dont occur due to dependencies incompatibility

In [1]:
# !pip install -q \
# numpy==1.26.4 \
# pandas==2.2.2 \
# datasets==2.19.1 \
# transformers==4.55.4 \
# accelerate \
# evaluate \
# sentencepiece \
# scikit-learn \
# peft

In [2]:
import os
import time
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)

import evaluate

In [3]:
!git clone https://github.com/nyu-mll/GLUE-baselines.git

Cloning into 'GLUE-baselines'...
remote: Enumerating objects: 891, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 891 (delta 1), reused 3 (delta 1), pack-reused 886 (from 1)
Receiving objects: 100% (891/891), 1.48 MiB | 2.34 MiB/s, done.
Resolving deltas: 100% (610/610), done.


In [4]:
%cd GLUE-baselines

!python download_glue_data.py \
--data_dir glue_data \
--tasks CoLA

/content/GLUE-baselines
	Completed!


In [5]:
!ls glue_data/CoLA

dev.tsv  original  test.tsv  train.tsv


Preparing the data

In [6]:
train_df = pd.read_csv(
    "glue_data/CoLA/train.tsv",
    sep="\t",
    header=None
)

dev_df = pd.read_csv(
    "glue_data/CoLA/dev.tsv",
    sep="\t",
    header=None
)

train_df = train_df[[1,3]]
dev_df = dev_df[[1,3]]

train_df.columns = ["label","sentence"]
dev_df.columns = ["label","sentence"]

In [7]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(dev_df)

MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

def tokenize_function(examples):

    return tokenizer(
        examples["sentence"],
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

val_dataset = val_dataset.rename_column(
    "label",
    "labels"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

for name, module in model.named_modules():

    if "query" in name.lower() or \
       "value" in name.lower():

        print(name)

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


deberta.encoder.layer.0.attention.self.query_proj
deberta.encoder.layer.0.attention.self.value_proj
deberta.encoder.layer.1.attention.self.query_proj
deberta.encoder.layer.1.attention.self.value_proj
deberta.encoder.layer.2.attention.self.query_proj
deberta.encoder.layer.2.attention.self.value_proj
deberta.encoder.layer.3.attention.self.query_proj
deberta.encoder.layer.3.attention.self.value_proj
deberta.encoder.layer.4.attention.self.query_proj
deberta.encoder.layer.4.attention.self.value_proj
deberta.encoder.layer.5.attention.self.query_proj
deberta.encoder.layer.5.attention.self.value_proj
deberta.encoder.layer.6.attention.self.query_proj
deberta.encoder.layer.6.attention.self.value_proj
deberta.encoder.layer.7.attention.self.query_proj
deberta.encoder.layer.7.attention.self.value_proj
deberta.encoder.layer.8.attention.self.query_proj
deberta.encoder.layer.8.attention.self.value_proj
deberta.encoder.layer.9.attention.self.query_proj
deberta.encoder.layer.9.attention.self.value_proj


LoRA:

In [9]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=[
        "query_proj",
        "value_proj"
    ]
)

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

In [10]:
# !pip uninstall -y torchao
# !pip install -U torchao

In [11]:
model = get_peft_model(
    model,
    lora_config
)

In [12]:
model.print_trainable_parameters()

trainable params: 296,450 || all params: 184,720,132 || trainable%: 0.1605


In [13]:
for name, param in model.named_parameters():
    if "lora" in name.lower():
        print(name)

base_model.model.deberta.encoder.layer.0.attention.self.query_proj.lora_A.default.weight
base_model.model.deberta.encoder.layer.0.attention.self.query_proj.lora_B.default.weight
base_model.model.deberta.encoder.layer.0.attention.self.value_proj.lora_A.default.weight
base_model.model.deberta.encoder.layer.0.attention.self.value_proj.lora_B.default.weight
base_model.model.deberta.encoder.layer.1.attention.self.query_proj.lora_A.default.weight
base_model.model.deberta.encoder.layer.1.attention.self.query_proj.lora_B.default.weight
base_model.model.deberta.encoder.layer.1.attention.self.value_proj.lora_A.default.weight
base_model.model.deberta.encoder.layer.1.attention.self.value_proj.lora_B.default.weight
base_model.model.deberta.encoder.layer.2.attention.self.query_proj.lora_A.default.weight
base_model.model.deberta.encoder.layer.2.attention.self.query_proj.lora_B.default.weight
base_model.model.deberta.encoder.layer.2.attention.self.value_proj.lora_A.default.weight
base_model.model.debe

In [14]:
metric = evaluate.load(
    "glue",
    "cola"
)

In [15]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(
        logits,
        axis=-1
    )

    return metric.compute(
        predictions=preds,
        references=labels
    )

In [16]:
training_args = TrainingArguments(
    output_dir="./lora_cola",

    learning_rate=2e-4,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=10,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="matthews_correlation",

    greater_is_better=True,

    logging_steps=50,

    report_to="none"
)

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

/tmp/ipykernel_10665/601936198.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Training

In [18]:
start = time.time()

trainer.train()

end = time.time()

training_time = (end - start) / 60

print(
    f"Training Time: {training_time:.2f} minutes"
)

Epoch,Training Loss,Validation Loss,Matthews Correlation
1,0.373200,0.351504,0.634099
2,0.335900,0.396946,0.625680
3,0.333300,0.360716,0.679967
4,0.279100,0.410012,0.659328
5,0.252600,0.406262,0.674901
6,0.274500,0.412591,0.674886
7,0.250300,0.398448,0.675169
8,0.209200,0.394930,0.675405
9,0.207400,0.400603,0.678003
10,0.223900,0.427415,0.670267


Training Time: 9.31 minutes


In [19]:
results = trainer.evaluate()

print(results)

{'eval_loss': 0.3607160747051239, 'eval_matthews_correlation': 0.6799667905057818, 'eval_runtime': 3.0912, 'eval_samples_per_second': 337.406, 'eval_steps_per_second': 21.351, 'epoch': 10.0}


In [20]:
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

lora_results = {
    "Method": "LoRA",
    "MCC": results["eval_matthews_correlation"],
    "Trainable_Params": trainable_params,
    "Rank": 8,
    "Training_Time_Min": training_time
}

print(lora_results)

{'Method': 'LoRA', 'MCC': 0.6799667905057818, 'Trainable_Params': 296450, 'Rank': 8, 'Training_Time_Min': 9.314110406239827}


In [21]:
import torch
import numpy as np

effective_ranks = []

for name, module in model.named_modules():

    if hasattr(module, "lora_A") and hasattr(module, "lora_B"):

        A = module.lora_A["default"].weight.data
        B = module.lora_B["default"].weight.data

        delta_w = B @ A

        s = torch.linalg.svdvals(delta_w)

        threshold = 0.01 * s.max()

        eff_rank = (s > threshold).sum().item()

        effective_ranks.append(eff_rank)

        print(f"{name}: effective rank = {eff_rank}")

print("\nAverage Effective Rank:", np.mean(effective_ranks))

base_model.model.deberta.encoder.layer.0.attention.self.query_proj: effective rank = 8
base_model.model.deberta.encoder.layer.0.attention.self.value_proj: effective rank = 8
base_model.model.deberta.encoder.layer.1.attention.self.query_proj: effective rank = 8
base_model.model.deberta.encoder.layer.1.attention.self.value_proj: effective rank = 8
base_model.model.deberta.encoder.layer.2.attention.self.query_proj: effective rank = 8
base_model.model.deberta.encoder.layer.2.attention.self.value_proj: effective rank = 8
base_model.model.deberta.encoder.layer.3.attention.self.query_proj: effective rank = 8
base_model.model.deberta.encoder.layer.3.attention.self.value_proj: effective rank = 8
base_model.model.deberta.encoder.layer.4.attention.self.query_proj: effective rank = 8
base_model.model.deberta.encoder.layer.4.attention.self.value_proj: effective rank = 8
base_model.model.deberta.encoder.layer.5.attention.self.query_proj: effective rank = 8
base_model.model.deberta.encoder.layer.5.at

In [22]:
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained("/content/drive/MyDrive/lora_cola_final")
tokenizer.save_pretrained("/content/drive/MyDrive/lora_cola_final")

Mounted at /content/drive


('/content/drive/MyDrive/lora_cola_final/tokenizer_config.json',
 '/content/drive/MyDrive/lora_cola_final/special_tokens_map.json',
 '/content/drive/MyDrive/lora_cola_final/spm.model',
 '/content/drive/MyDrive/lora_cola_final/added_tokens.json',
 '/content/drive/MyDrive/lora_cola_final/tokenizer.json')